In [63]:
import pandas as pd
from sqlalchemy import create_engine

In [64]:
path = "../data/eco2mix-national.csv"

In [65]:
df = pd.read_csv(path, sep=';')

In [66]:
df.dtypes

Périmètre                                        str
Nature                                           str
Date                                             str
Heure                                            str
Date - Heure                                     str
Consommation (MW)                            float64
Prévision J-1 (MW)                             int64
Prévision J (MW)                             float64
Fioul (MW)                                   float64
Charbon (MW)                                 float64
Gaz (MW)                                     float64
Nucléaire (MW)                               float64
Eolien (MW)                                  float64
Eolien terrestre (MW)                        float64
Eolien offshore (MW)                         float64
Solaire (MW)                                 float64
Hydraulique (MW)                             float64
Pompage (MW)                                 float64
Bioénergies (MW)                             f

In [67]:
print(pd.io.sql.get_schema(df, name='energy_data', con=engine))


CREATE TABLE energy_data (
	"Périmètre" TEXT, 
	"Nature" TEXT, 
	"Date" TEXT, 
	"Heure" TEXT, 
	"Date - Heure" TEXT, 
	"Consommation (MW)" FLOAT(53), 
	"Prévision J-1 (MW)" BIGINT, 
	"Prévision J (MW)" FLOAT(53), 
	"Fioul (MW)" FLOAT(53), 
	"Charbon (MW)" FLOAT(53), 
	"Gaz (MW)" FLOAT(53), 
	"Nucléaire (MW)" FLOAT(53), 
	"Eolien (MW)" FLOAT(53), 
	"Eolien terrestre (MW)" FLOAT(53), 
	"Eolien offshore (MW)" FLOAT(53), 
	"Solaire (MW)" FLOAT(53), 
	"Hydraulique (MW)" FLOAT(53), 
	"Pompage (MW)" FLOAT(53), 
	"Bioénergies (MW)" FLOAT(53), 
	"Ech. physiques (MW)" FLOAT(53), 
	"Taux de CO2 (g/kWh)" FLOAT(53), 
	"Ech. comm. Angleterre (MW)" FLOAT(53), 
	"Ech. comm. Espagne (MW)" FLOAT(53), 
	"Ech. comm. Italie (MW)" FLOAT(53), 
	"Ech. comm. Suisse (MW)" FLOAT(53), 
	"Ech. comm. Allemagne-Belgique (MW)" FLOAT(53), 
	"Fioul - TAC (MW)" FLOAT(53), 
	"Fioul - Cogénération (MW)" FLOAT(53), 
	"Fioul - Autres (MW)" FLOAT(53), 
	"Gaz - TAC (MW)" FLOAT(53), 
	"Gaz - Cogénération (MW)" FLOAT(53), 
	"G

## Ingestion

In [68]:
df.head(n=0).to_sql(name='energy_data', con=engine, if_exists='replace') ## creation of the table

0

In [69]:
import re
import unicodedata

def to_snake(name: str) -> str:
    name = re.sub(r'\s*\(.*?\)', '', name)  # retire les unités "(MW)"
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode()  # é → e
    name = re.sub(r'[^a-zA-Z0-9]+', '_', name)  # espaces, tirets, points, ' → _
    return name.strip('_').lower()

In [70]:
def transform(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns=to_snake)
    df['date_heure'] = pd.to_datetime(df['date_heure'], utc=True)
    df['date'] = pd.to_datetime(df['date']).dt.date
    return df.drop(columns=['heure'])

df_iter = pd.read_csv(path, sep=';', chunksize=200)

In [71]:
for i, chunk in enumerate(df_iter):
    chunk = transform(chunk)
    chunk.to_sql('energy_data', con=engine,
                 if_exists='replace' if i == 0 else 'append', index=False)
    print("Inserted:", len(df_chunk))

Inserted: 112
Inserted: 112
Inserted: 112
Inserted: 112
Inserted: 112
Inserted: 112
Inserted: 112
Inserted: 112
Inserted: 112
Inserted: 112
Inserted: 112
Inserted: 112
Inserted: 112
Inserted: 112
Inserted: 112
Inserted: 112
Inserted: 112
Inserted: 112
Inserted: 112
Inserted: 112
Inserted: 112
Inserted: 112
Inserted: 112
